In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import scipy
import scipy.signal as sig
import soundfile as sf


# ---------------------------
# Loading / basic prep
# ---------------------------
def load_mono(path, target_sr=None):
    y, sr = sf.read(path, always_2d=False)
    if y.ndim > 1:
        y = y.mean(axis=1)
    y = y.astype(np.float32)

    if target_sr is not None and sr != target_sr:
        g = np.gcd(sr, target_sr)
        up = target_sr // g
        down = sr // g
        y = sig.resample_poly(y, up, down).astype(np.float32)
        sr = target_sr

    # normalize to avoid dumb scale issues
    y = y / (np.max(np.abs(y)) + 1e-12)
    return y, sr


def save_wav(path, y, sr):
    sf.write(path, np.clip(y, -1.0, 1.0), sr)


# ---------------------------
# Delay estimation + alignment
# ---------------------------
def estimate_delay_samples(x, d, max_lag=4000):
    """
    Cross-correlation lag estimate.
    Returns lag where positive means x lags behind d and should be advanced.
    """
    n = min(len(x), len(d))
    x0 = x[:n]
    d0 = d[:n]

    # light highpass to reduce DC / slow drift effects
    b, a = sig.butter(2, 50, btype="highpass", fs=48000)
    try:
        x0 = sig.filtfilt(b, a, x0)
        d0 = sig.filtfilt(b, a, d0)
    except Exception:
        pass

    corr = sig.correlate(d0, x0, mode="full", method="fft")
    mid = len(corr) // 2
    lo = mid - max_lag
    hi = mid + max_lag + 1
    seg = corr[lo:hi]
    best = np.argmax(seg) + lo
    lag = best - mid
    return int(lag)


def apply_delay(x, lag):
    if lag == 0:
        return x
    if lag > 0:
        # x is delayed -> advance (drop first lag samples)
        return x[lag:]
    else:
        # x is early -> pad
        return np.pad(x, (abs(lag), 0))


def bandpass(y, sr, lo, hi, order=4):
    sos = scipy.signal.butter(order, [lo, hi], btype="bandpass", fs=sr, output="sos")
    return scipy.signal.sosfiltfilt(sos, y)


def estimate_delay_samples(x, d, sr, band=(14000, 18000), max_lag=5000):
    # band-limit first so correlation locks onto air components, not random junk
    x_f = bandpass(x, sr, band[0], band[1])
    d_f = bandpass(d, sr, band[0], band[1])

    corr = scipy.signal.correlate(d_f, x_f, mode="full", method="fft")
    mid = len(corr) // 2
    seg = corr[mid - max_lag : mid + max_lag + 1]
    lag = int(np.argmax(seg) - max_lag)
    return lag


# ---------------------------
# NLMS Adaptive Noise Canceller
# ---------------------------
def nlms_anc(x, d, L=512, mu=0.5, eps=1e-8):
    n = min(len(x), len(d))
    w = np.zeros(L, dtype=np.float32)
    y = np.zeros(n, dtype=np.float32)
    e = np.zeros(n, dtype=np.float32)

    xpad = np.zeros(L - 1 + n, dtype=np.float32)
    xpad[L - 1 :] = x[:n].astype(np.float32)
    d = d[:n].astype(np.float32)

    for i in range(n):
        xvec = xpad[i : i + L][::-1]
        y_i = np.dot(w, xvec)
        e_i = d[i] - y_i
        w += (mu / (np.dot(xvec, xvec) + eps)) * e_i * xvec
        y[i] = y_i
        e[i] = e_i

    return e, y


# ---------------------------
# Metrics / inspection
# ---------------------------
def erle_db(d, e, sr, t0_s, t1_s):
    """
    ERLE on a segment where milling is absent (air-only).
    Higher is better.
    """
    i0 = int(t0_s * sr)
    i1 = int(t1_s * sr)
    dseg = d[i0:i1]
    eseg = e[i0:i1]
    num = np.mean(dseg**2) + 1e-12
    den = np.mean(eseg**2) + 1e-12
    return 10.0 * np.log10(num / den)


def coherence_stats(x, d, e, sr, fmin=200, fmax=12000):
    f, c_before = sig.coherence(x, d, fs=sr, nperseg=4096)
    _, c_after = sig.coherence(x, e, fs=sr, nperseg=4096)
    band = (f >= fmin) & (f <= min(fmax, sr / 2 - 1))
    return float(np.mean(c_before[band])), float(np.mean(c_after[band])), f, c_before, c_after


def mean_coherence_in_band(x, y, sr, lo, hi):
    f, cxy = scipy.signal.coherence(x, y, fs=sr, nperseg=4096)
    m = (f >= lo) & (f <= hi)
    return float(np.mean(cxy[m])), f, cxy


def plot_spectrogram(y, sr, title, nperseg=2048, noverlap=1536, fmax=20000, vmin=None, vmax=None):
    f, t, S = sig.spectrogram(
        y,
        fs=sr,
        window="hann",
        nperseg=nperseg,
        noverlap=noverlap,
        scaling="spectrum",
        mode="magnitude",
    )
    S_db = 20 * np.log10(S + 1e-8)

    fmax_eff = min(fmax, sr / 2)
    mask = f <= fmax_eff
    f = f[mask]
    S_db = S_db[mask, :]

    plt.figure(figsize=(12, 4))
    plt.pcolormesh(t, f, S_db, shading="auto", vmin=vmin, vmax=vmax, cmap="magma")
    plt.ylim(0, fmax_eff)
    plt.xlabel("Zeit [s]")
    plt.ylabel("Frequenz [Hz]")
    plt.title(title)
    plt.colorbar(label="Magnitude [dB]")
    plt.tight_layout()


def plot_wave_snip(d, e, sr, t0=1.0, dur=0.05):
    i0 = int(t0 * sr)
    i1 = int((t0 + dur) * sr)
    tt = np.arange(i0, i1) / sr
    plt.figure(figsize=(12, 3))
    plt.plot(tt, d[i0:i1], label="Primärsignal (Luft + Fräsen)")
    plt.plot(tt, e[i0:i1], label="Bereinigtes Signal", alpha=0.85)
    plt.legend()
    plt.title("Signalausschnitt")
    plt.tight_layout()


def plot_coherence(f, c_before, c_after):
    plt.figure(figsize=(12, 3))
    plt.semilogx(f, c_before, label="Kohärenz(Referenz, Primär)")
    plt.semilogx(f, c_after, label="Kohärenz(Referenz, Bereinigt)")
    plt.ylim(0, 1.05)
    plt.xlabel("Frequenz [Hz]")
    plt.ylabel("Kohärenz")
    plt.title("Kohärenz vor und nach ANC")
    plt.legend()
    plt.tight_layout()

def plot_anc_gesamt(
    d_primary,
    cleaned,
    sr,
    f,
    c_before,
    c_after,
    *,
    gesamttitel="ANC-Auswertung",
    t0=1.0,
    dur=0.05,
    fmax=12000,
    vmin=-120,
    vmax=-20,
):
    import numpy as np
    import matplotlib.pyplot as plt
    from scipy import signal

    fig, axes = plt.subplots(
        4, 1, figsize=(10, 9), constrained_layout=True
    )

    # --------------------------------------------------
    # 1) Zeitsignal-Ausschnitt
    # --------------------------------------------------
    i0 = int(t0 * sr)
    i1 = int((t0 + dur) * sr)
    i1 = min(i1, len(d_primary), len(cleaned))

    tt = np.arange(i0, i1) / sr

    axes[0].plot(tt, d_primary[i0:i1], label="Primärsignal (Luft + Fräsen)", linewidth=1.0)
    axes[0].plot(tt, cleaned[i0:i1], label="Bereinigtes Signal", linewidth=1.0, alpha=0.9)
    axes[0].set_title("Zeitsignal-Ausschnitt")
    axes[0].set_xlabel("Zeit [s]")
    axes[0].set_ylabel("Amplitude")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    # --------------------------------------------------
    # 2) Primärsignal-Spektrogramm
    # --------------------------------------------------
    f_sp1, t_sp1, S1 = signal.spectrogram(
        d_primary,
        fs=sr,
        window="hann",
        nperseg=2048,
        noverlap=1536,
        scaling="spectrum",
        mode="magnitude",
    )
    S1_db = 20 * np.log10(S1 + 1e-12)
    m1 = f_sp1 <= fmax

    im1 = axes[1].pcolormesh(
        t_sp1,
        f_sp1[m1],
        S1_db[m1, :],
        shading="auto",
        cmap="magma",
        vmin=vmin,
        vmax=vmax,
    )
    axes[1].set_title("Primärsignal-Spektrogramm")
    axes[1].set_xlabel("Zeit [s]")
    axes[1].set_ylabel("Frequenz [Hz]")
    cbar1 = fig.colorbar(im1, ax=axes[1])
    cbar1.set_label("Magnitude [dB]")

    # --------------------------------------------------
    # 3) Bereinigtes Spektrogramm
    # --------------------------------------------------
    f_sp2, t_sp2, S2 = signal.spectrogram(
        cleaned,
        fs=sr,
        window="hann",
        nperseg=2048,
        noverlap=1536,
        scaling="spectrum",
        mode="magnitude",
    )
    S2_db = 20 * np.log10(S2 + 1e-12)
    m2 = f_sp2 <= fmax

    im2 = axes[2].pcolormesh(
        t_sp2,
        f_sp2[m2],
        S2_db[m2, :],
        shading="auto",
        cmap="magma",
        vmin=vmin,
        vmax=vmax,
    )
    axes[2].set_title("Bereinigtes Spektrogramm")
    axes[2].set_xlabel("Zeit [s]")
    axes[2].set_ylabel("Frequenz [Hz]")
    cbar2 = fig.colorbar(im2, ax=axes[2])
    cbar2.set_label("Magnitude [dB]")

    # --------------------------------------------------
    # 4) Kohärenz vorher / nachher
    # --------------------------------------------------
    axes[3].semilogx(f, c_before, label="Kohärenz(Referenz, Primär)", linewidth=1.5)
    axes[3].semilogx(f, c_after, label="Kohärenz(Referenz, Bereinigt)", linewidth=1.5)
    axes[3].set_title("Kohärenz vor und nach ANC")
    axes[3].set_xlabel("Frequenz [Hz]")
    axes[3].set_ylabel("Kohärenz")
    axes[3].set_ylim(0, 1.02)
    axes[3].grid(True, which="both", alpha=0.3)
    axes[3].legend()

    fig.suptitle(gesamttitel, fontsize=15, x=0.5, ha="center")

    return fig, axes

In [ ]:
if __name__ == "__main__":
    # luftdominantes Mikrofon (Referenz)
    ref_path = "13_01/only_air_front_1.flac"
    # fräsdominantes Mikrofon (Primärsignal)
    pri_path = "13_01/4mm_mit_Luft_front_2.flac"

    # Ursprüngliche Abtastrate beibehalten (192 kHz)
    d_primary, sr = load_mono(pri_path, target_sr=192000)
    x_ref, _ = load_mono(ref_path, target_sr=sr)

    # --- Parameter ---
    LAG_BAND = (8000, 14000)
    MAX_LAG_SAMPLES = 5000

    ANC_BAND = (1800, 6500)

    L = 512
    MU = 0.05
    EPS = 1e-8

    # --- Kanäle zeitlich ausrichten ---
    lag = estimate_delay_samples(x_ref, d_primary, sr=sr, band=LAG_BAND, max_lag=MAX_LAG_SAMPLES)
    print(f"Geschätzte Verzögerung: {lag} Samples = {lag / sr:.6f} s")

    if lag > 0:
        x_ref = x_ref[lag:]
        d_primary = d_primary[: len(x_ref)]
    elif lag < 0:
        d_primary = d_primary[-lag:]
        x_ref = x_ref[: len(d_primary)]

    # Sicherheitshalber auf gemeinsame Länge kürzen
    n = min(len(x_ref), len(d_primary))
    x_ref = x_ref[:n]
    d_primary = d_primary[:n]

    # --- Bandbegrenzung für stabile ANC-Anpassung ---
    x_adapt = bandpass(x_ref, sr, *ANC_BAND)
    d_adapt = bandpass(d_primary, sr, *ANC_BAND)

    # --- ANC (NLMS) wirklich anwenden ---
    # y_band = geschätzter luftdominanter Störanteil im Primärsignal
    # e_band = Fehler-/Restsignal im ANC-Band
    e_band, y_band = nlms_anc(x_adapt, d_adapt, L=L, mu=MU, eps=EPS)

    # Geschätzten Störanteil vom Vollbandsignal abziehen
    cleaned = d_primary - y_band

    # save_wav("cleaned_milling_est.wav", cleaned, sr)
    # save_wav("estimated_air_noise_band.wav", y_band, sr)

    # --- Kennzahlen ---
    try:
        erle = erle_db(d_adapt, e_band, sr, t0_s=2.0, t1_s=11.5)
        print(f"ERLE im luftdominierten Segment (Band): {erle:.2f} dB")
    except Exception:
        print("ERLE übersprungen. Falls nötig ein bekanntes luftdominiertes Segment wählen.")

    cb_mean, f, c_before = mean_coherence_in_band(x_adapt, d_adapt, sr, *ANC_BAND)
    ca_mean, _, c_after = mean_coherence_in_band(x_adapt, e_band, sr, *ANC_BAND)
    print(f"Mittlere Kohärenz im ANC-Band vorher: {cb_mean:.4f}")
    print(f"Mittlere Kohärenz im ANC-Band nachher: {ca_mean:.4f}")

    # --- Plots ---
    PLOT_TITEL = "ANC-Auswertung | 4 mm Nut | Bearbeitung mit Luft"

    fig, axes = plot_anc_gesamt(
        d_primary,
        cleaned,
        sr,
        f,
        c_before,
        c_after,
        gesamttitel=PLOT_TITEL,
        t0=1.0,
        dur=0.05,
        fmax=12000,
        vmin=-120,
        vmax=-20,
    )
    fig.savefig("anc_gesamtplot.png", dpi=300, bbox_inches="tight")

    plt.show()